In [15]:
import numpy as np
import pandas as pd

In [16]:
import os
print(os.path.exists('../data/processed/train_optimized.pkl'))

True


In [17]:
train = pd.read_pickle('../data/processed/train_optimized.pkl')
print(train.shape)
print(f'{train.memory_usage(deep=True).sum():,}')

(590540, 434)
953,877,651


In [18]:
train = train.copy()

In [19]:
train['trans_day'] = (train['TransactionDT'] // 86400).astype(np.int16)
train['trans_hour'] = ((train['TransactionDT'] % 86400) // 3600).astype(np.int8)
train['trans_weekday'] = (train['trans_day'] % 7).astype(np.int8)

In [20]:
train = train.sort_values('TransactionDT').reset_index(drop=True)
train['uid'] = train['card1'].astype(str) + '_' + train['addr1'].astype(str)
print(train['uid'].nunique())
print(train[['TransactionDT', 'card1', 'addr1', 'uid']].head(10))

37531
   TransactionDT  card1  addr1          uid
0          86400  13926  315.0  13926_315.0
1          86401   2755  325.0   2755_325.0
2          86469   4663  330.0   4663_330.0
3          86499  18132  476.0  18132_476.0
4          86506   4497  420.0   4497_420.0
5          86510   5937  272.0   5937_272.0
6          86522  12308  126.0  12308_126.0
7          86529  12695  325.0  12695_325.0
8          86535   2803  337.0   2803_337.0
9          86536  17399  204.0  17399_204.0


In [24]:
uid_grp = train.groupby('uid')['TransactionAmt']
train['uid_transaction_count'] = uid_grp.cumcount()
train['uid_mean_amount'] = uid_grp.transform(lambda x: x.shift(1).expanding().mean())
train['uid_std_amount'] = uid_grp.transform(lambda x: x.shift(1).expanding().std())
train['uid_amount_ratio'] = train['TransactionAmt']/train['uid_mean_amount']
train['uid_amount_zscore'] = (train['TransactionAmt'] - train['uid_mean_amount']) / (train['uid_std_amount'])
train['uid_mean_hour'] = train.groupby('uid')['trans_hour'].transform(lambda x: x.shift(1).expanding().mean())
print(train[['uid', 'TransactionAmt', 'uid_transaction_count', 'uid_mean_amount', 'uid_std_amount', 'uid_amount_ratio', 'uid_amount_zscore']].head(20))

            uid  TransactionAmt  uid_transaction_count  uid_mean_amount  \
0   13926_315.0       68.500000                    0.0              NaN   
1    2755_325.0       29.000000                    0.0              NaN   
2    4663_330.0       59.000000                    0.0              NaN   
3   18132_476.0       50.000000                    0.0              NaN   
4    4497_420.0       50.000000                    0.0              NaN   
5    5937_272.0       49.000000                    0.0              NaN   
6   12308_126.0      159.000000                    0.0              NaN   
7   12695_325.0      422.500000                    0.0              NaN   
8    2803_337.0       15.000000                    0.0              NaN   
9   17399_204.0      117.000000                    0.0              NaN   
10          NaN       75.887001                    NaN              NaN   
11          NaN       16.495001                    NaN              NaN   
12   3786_204.0       50.

In [25]:
uid_counts = train['uid'].value_counts()
print(uid_counts.head(10))

top_uid = uid_counts.index[0]
print(train[train['uid'] == top_uid][['uid', 'TransactionDT', 'TransactionAmt',
    'uid_transaction_count', 'uid_mean_amount',
    'uid_amount_ratio', 'uid_amount_zscore']].head(15))

uid
17188_299.0    5885
12695_325.0    5774
9500_204.0     4665
12839_264.0    3547
16132_299.0    3537
15497_299.0    3426
9500_272.0     2725
7664_264.0     2552
7207_204.0     2327
9112_441.0     2290
Name: count, dtype: int64
             uid  TransactionDT  TransactionAmt  uid_transaction_count  \
65   17188_299.0          87650      114.949997                    0.0   
81   17188_299.0          87868      104.949997                    1.0   
89   17188_299.0          88042      318.950012                    2.0   
159  17188_299.0          89072       92.949997                    3.0   
163  17188_299.0          89103      973.950012                    4.0   
238  17188_299.0          90163      226.000000                    5.0   
293  17188_299.0          91119      117.000000                    6.0   
394  17188_299.0          92760      344.950012                    7.0   
409  17188_299.0          93055      210.949997                    8.0   
421  17188_299.0          9335

In [26]:
print(f"NaN uids: {train['uid'].isna().sum()}")
train['uid'] = train['uid'].fillna('unknown_addr')
print(f"NaN uids after fix: {train['uid'].isna().sum()}")

NaN uids: 65706
NaN uids after fix: 0


In [27]:
train['uid_mean_amount'] = train['uid_mean_amount'].fillna(-1)
train['uid_std_amount'] = train['uid_std_amount'].fillna(-1)
train['uid_amount_ratio'] = train['uid_amount_ratio'].fillna(-1)
train['uid_amount_zscore'] = train['uid_amount_zscore'].fillna(-1)
train['uid_mean_hour'] = train['uid_mean_hour'].fillna(-1)
print(train[['uid_mean_amount', 'uid_std_amount', 'uid_amount_ratio',
             'uid_amount_zscore', 'uid_mean_hour']].isnull().sum())

uid_mean_amount      0
uid_std_amount       0
uid_amount_ratio     0
uid_amount_zscore    0
uid_mean_hour        0
dtype: int64
